In [ ]:
import os
import glob
import sys
from pathlib import Path
from astropy.io import fits
import matplotlib.pyplot as plt 
import pandas as pd 
import numpy as np 

l0_l1b_parent = Path("/home/bekah/m3-pipeline-dev")
if str(l0_l1b_parent) not in sys.path:
    sys.path.insert(0, str(l0_l1b_parent))
    
from l0_l1b_l2.reference import check_l1b_label, check_observation 


In [ ]:
#obs_id = "m3g20090111t013904"  
obs_id = "m3g20090811t135702"

line = 6789


In [ ]:
warn, error, metadata = check_observation(obs_id) 

In [ ]:
print(f"obs temp: {metadata['obs_temperature']}, dark temp {metadata['dark_signal_temp']}")

In [ ]:
metadata

In [ ]:
dark_id = metadata['dark_signal_id'].lower()

flat_id = metadata['flat_field_id'].lower()

bde_id = metadata['bad_detector_map_id'].lower()

l0_DN = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l0.fits"

l1b_result = og = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l1b_rdn.fits"

l1b_label = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l1b.xml" 

dark = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{dark_id}_l0.fits"

ff = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{flat_id}_ff.fits" 

lab_ff = f"/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global.fits"

bde =  f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{bde_id}_bde.fits" 

ssc = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_ssc.txt"

rdn_cal_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/cal_data/m3g20081118_rdn_cal.tab"

In [ ]:
# load each fits file 

with fits.open(l0_DN) as hdul1:
    l0 = hdul1[0].data
    
with fits.open(l1b_result) as hdul1:
    l1b = hdul1[0].data


In [ ]:
# basic flats. if you want. 

fits.writeto(
            f"l0_{obs_id}_flat.fits",
            np.mean(l0-dark_med[:,np.newaxis,:], axis=1)/np.mean(l0-dark_med[:,np.newaxis,:], axis=(1,2))[:, np.newaxis], 
            overwrite=True
        )

fits.writeto(
            f"l0_{obs_id}_medflat.fits",
            np.median(l0-dark_med[:,np.newaxis,:], axis=1)/np.mean(l0-dark_med[:,np.newaxis,:], axis=(1,2))[:, np.newaxis], 
            overwrite=True
        )

In [ ]:
# orient l1b to match l0 

reverse_lines, reverse_samples = check_l1b_label(l1b_label)

if reverse_lines:
    l1b = l1b[:, ::-1, :]
if reverse_samples:
    l1b = l1b[:, :, ::-1]

# fits.writeto(
#             f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l1b_flipped.fits",
#             l1b, 
#             overwrite=True
#         )

In [ ]:
# keep just the chosen line for l0 and l1b 

l0_line = l0[:, line, :] 
l1b_line = l1b[:, line, :] 

del l0 
del l1b

In [ ]:
# dark signal options 

with fits.open(dark) as hdul1:
    dark = hdul1[0].data.transpose(1,0,2)

dark_med = np.nanmedian(dark[5:-5, :, :], axis=0)
dark_mean = np.nanmean(dark[5:-5, :, :], axis=0)
dark_mean_notrim = np.nanmean(dark, axis=0)         # keeping weirdness at beginning and ends 

del dark 

In [ ]:
# flats 

with fits.open(lab_ff) as hdul1:
    labff = hdul1[0].data

with fits.open(ff) as hdul1:
    obsff = hdul1[0].data


In [ ]:
# rdn cal 
cal = pd.read_fwf(
        rdn_cal_path,
        names=['channel', 'rdn_cal_coeff'])
rdn_cals = cal['rdn_cal_coeff'].values
# obs_image = obs_image * rdn_cal[:, np.newaxis, np.newaxis]


In [ ]:
# smooth shape correction 
ssc_table = pd.read_fwf(ssc, names=["channel", "corr_factor"])
ssc = ssc_table['corr_factor'].values
#     obs_image = obs_image * ssc_factors[:, np.newaxis, np.newaxis]


In [ ]:
# trim 
def trim(frame):
    
    omitted_channels = [0]
    left_col_cutoff = 9
    right_col_cutoff = 313
    
    return frame[
                np.max(omitted_channels) + 1:,
                left_col_cutoff:right_col_cutoff
                ]


In [ ]:
# dark pedestal related things 

dark_ped = l0_line - dark_mean

all_bands = range(86) 

dark_cols = [1, 2, 3, 318, 319] 
plt.plot(all_bands, dark_ped[:, 1]) 
plt.plot(all_bands, dark_ped[:, 2]) 
plt.plot(all_bands, dark_ped[:, 3]) 
plt.plot(all_bands, dark_ped[:, 319]) 
plt.plot(all_bands, dark_ped[:, 318]) 

per_band_med = np.median(dark_ped[:, dark_cols], axis=1)
per_band_mean = np.mean(dark_ped[:, dark_cols], axis=1)
all_band_med = np.median(dark_ped[:, dark_cols])
all_band_mean = np.mean(dark_ped[:, dark_cols])

plt.plot(all_bands, per_band_med, color='red', ls=':')
plt.plot(all_bands, per_band_mean, color='green', ls=':')
plt.axhline(all_band_mean, c='green')
plt.axhline(all_band_med, c='red')

plt.ylim(-10, 10)

In [ ]:
# scattered light related things 

dark_ped = l0_line - dark_mean + abs(all_band_mean) 

all_bands = range(86) 

sl_cols = [4, 5, 6, 7,  314, 315, 316, 317] 
plt.plot(all_bands, dark_ped[:, 4]) 
plt.plot(all_bands, dark_ped[:, 5]) 
plt.plot(all_bands, dark_ped[:, 6]) 
# plt.plot(all_bands, dark_ped[:, 313]) 
plt.plot(all_bands, dark_ped[:, 314]) 
plt.plot(all_bands, dark_ped[:, 315]) 
plt.plot(all_bands, dark_ped[:, 316]) 
plt.plot(all_bands, dark_ped[:, 317]) 

sl_per_band_med = np.median(dark_ped[:, sl_cols], axis=1)
sl_per_band_mean = np.mean(dark_ped[:, sl_cols], axis=1)
sl_all_band_med = np.median(dark_ped[:, sl_cols])
sl_all_band_mean = np.mean(dark_ped[:, sl_cols])

plt.plot(all_bands, sl_per_band_med, color='red', ls=':')
plt.plot(all_bands, sl_per_band_mean, color='green', ls=':')
plt.axhline(sl_all_band_mean, c='green')
plt.axhline(sl_all_band_med, c='red')


In [ ]:
plt.plot(all_bands, sl_per_band_mean/np.mean(dark_ped, axis=1))
plt.plot(all_bands, sl_per_band_med/np.median(dark_ped, axis=1), c='red')
plt.axhline(.05)
plt.axhline(.01)

In [ ]:
# global_sl_ratios = np.array([
#   0.61, 0.52, 0.09, 0.09, 0.1, 0.1, 0.1, 0.1, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
#   0.02, 0.02, 0.02, 0.02, 0.0, 0.14
# ])

global_sl_ratios = np.array([
  0.61, 0.57, 0.09, 0.09, 0.06, 0.06, 0.1, 0.1, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06,
  0.02, 0.02, 0.02, 0.02, 0.0, 0.2
])


# global_sl_ratios = np.array([0.52643372, 0.318916  , 0.20992135, 0.1172837 , 0.10120929,
#        0.11565821, 0.11879554, 0.08658538, 0.09427511, 0.10029158,
#        0.11899142, 0.12910075, 0.1458139 , 0.12118314, 0.12523419,
#        0.1269864 , 0.10887364, 0.0793121 , 0.09968195, 0.07653839,
#        0.06932957, 0.06356107, 0.05156667, 0.05101482, 0.04794987,
#        0.03346814, 0.03675687, 0.02661979, 0.02784972, 0.02381307,
#        0.02482994, 0.01585399, 0.02883935, 0.02250688, 0.02356275,
#        0.02809257, 0.03324527, 0.03923005, 0.02150721, 0.02733526,
#        0.028926  , 0.03461809, 0.03409225, 0.0352231 , 0.03859862,
#        0.04065037, 0.02162314, 0.03339781, 0.03309865, 0.06496075,
#        0.02720287, 0.02222179, 0.02041227, 0.01171176, 0.01435737,
#        0.01864088, 0.0152631 , 0.01014737, 0.0135376 , 0.00909639,
#        0.00821105, 0.00736671, 0.00179099, 0.00745321, 0.00928469,
#        0.0044615 , 0.00898238, 0.00579344, 0.0034326 , 0.00794727,
#        0.00939988, 0.00882548, 0.01112269, 0.01509108, 0.01119793,
#        0.00742456, 0.00304756, 0.01489753, 0.01253894, 0.02366843,
#        0.0206263 , 0.02416508, 0.02772052, 0.02243191, 0.02963544,
#        0.05206255])

def basic_kernel_scattered_light_corr(
        band: int, 
        obs_band: np.ndarray,
        sl_ratio: float,
        sigma: float = 6.0,
):
    """
    Gaussian kernel applied per line per channel, scaled by the scattered light
    ratio per channel. Resulting image then scaled to retain removed scattered
    signal in peak areas.

    Args:
        obs_band: Obs image data, per channel.
        sl_ratio: Percent observed signal that is scattered light, based on
            signal in vignetted columns vs observing columns. We could evaluate
            this per line? But right now it's a set value for each band.
        sigma: Sigma for Gaussian kernel.
    """
    from scipy.ndimage import gaussian_filter1d

    
    # to take the image back to all signal retained after sl subtraction
    norm_factor = (1.0 / (1.0 - sl_ratio)) + sl_ratio
    scatter = gaussian_filter1d(obs_band, sigma=sigma)
    background = sl_ratio * scatter

    obs_band_new = (obs_band - background) * norm_factor

    plt.figure(figsize=(10, 6))
    plt.plot(range(320), obs_band, label='old') 
    plt.plot(range(320), obs_band_new, label='new') 
    plt.savefig(f"band{band}.png") 
    plt.close()
    
    return obs_band_new

def apply_scattered_light_corr(
        obs_image: np.ndarray,
        obs_type: str,
        sl_ratio_corr: bool = True,
        sigma: float = 5.0
):
    """
    Apply gauss kernel scattered light correction per band with correct
    SL ratio per band.

    TODO: flat subtraction of lowest or mean SL vignetted col value?

    Args:
        obs_image: Obs image data, preferably dark pedestal and ghost
            corrected.
        obs_type: Target or global mode.
        sl_ratio_corr: True / False if we want to replace negative correction
            factors with 0. I don't think we should really have these in the
            future but for now we do this.
        sigma: Sigma for Gaussian kernel.
    """
    bands = obs_image.shape[0]

    ratios = global_sl_ratios.copy() if obs_type.upper() == 'G' \
        else target_sl_ratios.copy()
    ratios = ratios #+ add
    if sl_ratio_corr:
        # no reason the correction ratio should ever be negative
        ratios[ratios < 0.0] = 0.0

    if len(ratios) != bands:
        raise ValueError(
            f"Scattered light ratio array for obs_type={obs_type} has "
            f"{len(ratios)} values but {bands} bands in the image."
        )

    for band in range(bands):
        sl_ratio = ratios[band]
        # modify in place atm, but maybe we want to make a copy?
        obs_image[band, :] = basic_kernel_scattered_light_corr(
            band=band,
            obs_band=obs_image[band, :],
            sl_ratio=sl_ratio,
            sigma=sigma,
        )
    return obs_image    

In [ ]:
# basic calibration 

#TODO:  need to add ghosting in bc that's actually significant at the 1-3 diff DN level 

# where l0_line, [band, col] 

r0 = trim((l0_line - dark_mean) * labff * rdn_cals[:, np.newaxis])

r1 = trim((l0_line - dark_mean) * labff / obsff * rdn_cals[:, np.newaxis])

r2 = trim((l0_line - dark_mean) * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

r3 = trim((l0_line - dark_mean_notrim)  * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

# diff dark pedestal options 
dss = l0_line - dark_mean 


r4 = trim((l0_line - dark_mean - per_band_mean[:, np.newaxis])  * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

# r5 = trim((l0_line - dark_mean  - per_band_med[:, np.newaxis])  * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

r6  = trim((l0_line - dark_mean  - all_band_med)  * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis]

r5 = trim(apply_scattered_light_corr(dss  - per_band_med[:, np.newaxis], 'G', sigma=5) * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

r7  = trim(apply_scattered_light_corr(l0_line - dark_mean - all_band_mean, 'G')  * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

r8 = trim((dss + np.mean(dss, axis=1)[:, np.newaxis] *.05 + sl_per_band_med[:, np.newaxis]) * labff / obsff * rdn_cals[:, np.newaxis]) *ssc[: , np.newaxis] 

r9 = trim(apply_scattered_light_corr(dss + np.median(dss, axis=1)[:, np.newaxis] *.05, 'G', sigma=5) * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

r10 = trim(apply_scattered_light_corr(dss + np.median(dss, axis=1)[:, np.newaxis] *.03, 'G', sigma=5) * labff / obsff * rdn_cals[:, np.newaxis])*ssc[: , np.newaxis] 

In [ ]:
# plot them 

bands = range(85)

col = 11

plt.figure(figsize=(10, 6))

plt.title("Radiance Comparison") 

options = [
       (l1b_line, 'l1b', 'black'),
       (r0, 'base', 'grey') , 
       (r1, 'obs ff', 'purple'), 
       # (r2, 'ssc', 'lime'), 
       # (r3, 'dark med', 'pink'), 
       # (r4, 'bmndp', 'orange'), 
       (r5, 'bmddp', 'orange'), 
       (r6, 'abmddp', 'green'), 
       (r7, 'sl ker', 'yellow'),
       # (r8, 'dp 5% sl col', 'brown'),
       (r9, 'dp 5% sl ker', 'red'), 
       (r10, 'dp 5% sl ker 10 sig', 'cyan'),
          ]

for i, lab, color in options: 
    plt.plot(bands, i[:, col], label=lab, c=color) 
plt.legend()

In [ ]:
# in DN 

plt.figure(figsize=(12, 8))

plt.title("DN Comparison") 

options = [
       (l1b_line, 'l1b', 'black'),
       # (r0, 'base', 'grey') , 
       # (r1, 'obs ff', 'purple'), 
       # (r2, 'ssc', 'lime'), 
       # (r3, 'dark med', 'pink'), 
       # (r4, 'bmndp', 'orange'), 
       (r5, 'bmddp', 'orange'), 
       # (r6, 'abmddp', 'green'), 
       # (r7, 'sl ker', 'yellow'),
       (r8, 'dp 5% sl col', 'brown'),
       (r9, 'dp 5% sl ker', 'red'), 
        (r10, 'dp 5% sl ker 10 sig', 'cyan'),
    
          ]

for i, lab, color in options: 
    plt.plot(bands, i[:, col]/ rdn_cals[1:], label=lab, c=color) 
plt.legend()

In [ ]:
plt.figure(figsize=(12, 8))
plt.grid()
for col in range(304):
    plt.plot(bands, ((r5[:, col]/ rdn_cals[1:]) - (l1b_line[:, col]/ rdn_cals[1:])), alpha=.1) 
plt.ylim(-10,10)

In [ ]:
plt.plot(bands, np.mean((r10[:, :]/ rdn_cals[1:, np.newaxis]) - (l1b_line[:, :]/ rdn_cals[1:, np.newaxis]), axis=1)) 


In [ ]:
plt.plot(bands, np.mean((r10[:, :]/ rdn_cals[1:, np.newaxis]) - (l1b_line[:, :]/ rdn_cals[1:, np.newaxis]))/np.median(r10[:, :]/ rdn_cals[1:, np.newaxis], axis=1)) 


In [ ]:
 add = np.mean((r10[:, :]/ rdn_cals[1:, np.newaxis]) - (l1b_line[:, :]/ rdn_cals[1:, np.newaxis]))/np.median(r10[:, :]/ rdn_cals[1:, np.newaxis], axis=1)

In [ ]:
add = np.insert(add, 0, 0)
add

In [ ]:
for col in range(304):
    plt.plot(bands, ((r9[:, col]/ rdn_cals[1:]) / (l1b_line[:, col]/ rdn_cals[1:])), alpha=.1) 
plt.ylim(0,3)

In [ ]:
for col in range(r7.shape[1]):
    plt.scatter(col, (l1b_line[0, col]/ rdn_cals[1])-(r10[0, col]/ rdn_cals[1])*1.2)
    
plt.axhline(np.mean((l1b_line[0, :]/ rdn_cals[1, np.newaxis])), color='green')
plt.axhline(np.mean((r10[0, :]/ rdn_cals[1, np.newaxis])), color='red')
            